In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'revision_work':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'revision_work' / 'Updated_data_aug24'
OUT_DIR = ROOT / 'revision_work' / 'data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'cohort':      'Participant_Status_24Aug2026.csv',
    'demographics':'Demographics_24Aug2026.csv',
    'family':      'Family_History_24Aug2026.csv',
    'diagnosis':   'PD_Diagnosis_History_24Aug2026.csv',
    'dopamine':    'Initiation_of_Dopaminergic_Therapy_24Aug2026.csv',
    'moca':        'Montreal_Cognitive_Assessment__MoCA__24Aug2026.csv',
    'falls':       'Determination_of_Freezing_and_Falls_24Aug2026.csv',
    'scopa':       'SCOPA-AUT_24Aug2026.csv',
    'neuroqol':    'Neuro_QoL__Lower_Extremity_Function__Mobility__-_Short_Form_24Aug2026.csv',
    'gds':         'Geriatric_Depression_Scale__Short_Version__24Aug2026.csv',
    'updrs1':      'MDS-UPDRS_Part_I_24Aug2026.csv',
    'updrs1q':     'MDS-UPDRS_Part_I_Patient_Questionnaire_24Aug2026.csv',
    'updrs3':      'MDS-UPDRS_Part_III_24Aug2026.csv',
    'updrs4':      'MDS-UPDRS_Part_IV__Motor_Complications_24Aug2026.csv',
    'vitals':      'Vital_Signs_24Aug2026.csv',
    'otherclin':   'Other_Clinical_Features_24Aug2026.csv',
}

pd.set_option('display.max_columns', None)
print('Data directory:', DATA_DIR)

Data directory: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/PD Prediction Old/Mobility_Decline_Risk_Prediction_For_PD_Patients-explanation_llm_v2/revision_work/Updated_data_aug24


## Helpers

Every longitudinal PPMI table carries a visit date in `INFODT` (`MM/YYYY`), so one generic accessor covers all of them.

`latest_at_index` is the single place the temporal rule lives: keep only rows dated at or before the patient's index date, then take the last non-missing value of each requested column. Columns are resolved independently, so a patient with MoCA at one visit and UPDRS at another keeps both.

In [2]:
def load_visits(key):
    """Load a longitudinal PPMI table and parse its visit date."""
    df = pd.read_csv(DATA_DIR / FILES[key], low_memory=False)
    df['DATE'] = pd.to_datetime(df['INFODT'], format='%m/%Y', errors='coerce')
    return df


def _last_valid(series):
    valid = series.dropna()
    return valid.iloc[-1] if len(valid) else np.nan


def _delta(series):
    valid = series.dropna()
    if len(valid) >= 2:
        return valid.iloc[-1] - valid.iloc[-2]
    return np.nan


OUTCOME_WINDOW_MONTHS = 12


def index_dates_for(outcome_visits, offset_months=OUTCOME_WINDOW_MONTHS):
    out = outcome_visits[['PATNO', 'outcome_date']].copy()
    out['index_date'] = out['outcome_date'] - pd.DateOffset(months=offset_months)
    return out


def latest_at_index(visits, cols, index_dates, inclusive=True):
    m = visits.merge(index_dates[['PATNO', 'index_date']], on='PATNO', how='inner')
    eligible = m['DATE'] <= m['index_date'] if inclusive else m['DATE'] < m['index_date']
    m = m[m['DATE'].notna() & eligible]
    
    LAST_COLS = ['MCATOT', 'GDS_TOTAL', 'BMI', 'able_weighted_score', 
                 'NP3GAIT_missing_due_to_101', 'NP3PSTBL_missing_due_to_101', 'NHY_missing_due_to_101']
    DELTA_COLS = ['MCATOT', 'NP1RTOT', 'NP3TOT', 'NP4TOT', 'GDS_TOTAL', 
                  'NP3GAIT', 'NP3PSTBL', 'NHY', 'BMI', 'able_weighted_score']
    
    # Pre-group by DATE to resolve same-day multi-visit conflicts (e.g. UPDRS ON/OFF)
    # We take the max (worst) score for that specific day
    m = m.groupby(['PATNO', 'DATE'])[cols].max().reset_index()
    
    agg_dict = {}
    for c in cols:
        if c in LAST_COLS:
            agg_dict[c] = _last_valid
        else:
            agg_dict[c] = 'max'
            
    out = (
        m.sort_values(['PATNO', 'DATE'])
         .groupby('PATNO')
         .agg(agg_dict)
         .reset_index()
    )
    
    deltas = []
    for c in cols:
        if c in DELTA_COLS:
            delta_series = m.sort_values(['PATNO', 'DATE']).groupby('PATNO')[c].agg(_delta).rename(f'delta_{c}')
            deltas.append(delta_series)
            
    if deltas:
        delta_df = pd.concat(deltas, axis=1).reset_index()
        out = out.merge(delta_df, on='PATNO', how='left')
        
    return out


## Cohort, outcome visit, index date

Cohort filter is unchanged from the legacy pipeline: PPMI `COHORT == 1` (Parkinson's disease) with an enrolled, complete, or deceased-withdrawal status.

This replaces `pick_row_for_patient()`, which searched a 3-year window for the visit with the *highest* fall count and then used freezing-of-gait from that same visit as a predictor. That is the specific leak the reviewer flagged. Here the outcome visit is simply the last one where falls were actually recorded, and nothing from that visit is used as a predictor.

Outcome recoding is unchanged: raw `FLNFR12M` classes 3 and 4 collapse into 2, giving no fall / rare fall / recurrent fall.

In [3]:
cohort_raw = pd.read_csv(DATA_DIR / FILES['cohort'])
pd_patients = (
    cohort_raw[
        (cohort_raw['COHORT'] == 1)
        & cohort_raw['ENROLL_STATUS'].isin(['Enrolled', 'Complete', 'Withdraw Deceased'])
    ][['PATNO']]
    .drop_duplicates()
)

falls = load_visits('falls')
falls = falls[falls['PATNO'].isin(pd_patients['PATNO'])]
falls_observed = falls[falls['FLNFR12M'].notna() & falls['DATE'].notna()]

outcome_visits = (
    falls_observed.sort_values(['PATNO', 'DATE'])
    .groupby('PATNO')
    .tail(1)[['PATNO', 'DATE', 'FLNFR12M']]
    .rename(columns={'DATE': 'outcome_date', 'FLNFR12M': 'falls_raw'})
    .reset_index(drop=True)
)
outcome_visits['falls_class'] = outcome_visits['falls_raw'].clip(upper=2).astype(int)

n_falls_visits = falls_observed.groupby('PATNO').size().rename('n_falls_visits')
outcome_visits = outcome_visits.merge(n_falls_visits, on='PATNO')

print(f'PD cohort                    : {len(pd_patients)}')
print(f'With an observed falls score : {len(outcome_visits)}')
print('\nOutcome distribution at outcome visit:')
print(outcome_visits['falls_class'].value_counts().sort_index()
      .rename({0: 'No fall', 1: 'Rare fall', 2: 'Recurrent fall'}))

PD cohort                    : 1362
With an observed falls score : 1280

Outcome distribution at outcome visit:
falls_class
No fall           869
Rare fall         288
Recurrent fall    123
Name: count, dtype: int64


## Patient-level (static) features

Demographics, family history, and diagnosis history are recorded once per patient, so there is no temporal choice to make — but the two *derived* fields do have one. Legacy computed `Age` and `No_of_years` relative to *today*, which for a patient whose last visit was in 2018 overstates both by several years. Both are now computed as of the index date, i.e. at the moment the prediction is made, not at the outcome visit a year later.

Dopaminergic therapy does carry a visit date, so it is treated as time-varying and read at or before the index date like any other predictor.

In [4]:
RACE_COLS = ['RAASIAN', 'RABLACK', 'RAHAWOPI', 'RAINDALS', 'RANOS', 'RAWHITE', 'RAUNKNOWN']
RACE_LABELS = {
    'RAASIAN': 'Asian',
    'RABLACK': 'Black or African American',
    'RAHAWOPI': 'Native Hawaiian or Other Pacific Islander',
    'RAINDALS': 'American Indian or Alaska Native',
    'RANOS': 'Not Otherwise Specified',
    'RAWHITE': 'White',
    'RAUNKNOWN': 'Unknown',
}


def infer_race(row):
    flagged = [RACE_LABELS[c] for c in RACE_COLS if pd.notna(row.get(c)) and float(row[c]) == 1.0]
    if not flagged:
        return 'Unknown'
    return flagged[0] if len(flagged) == 1 else 'Multiple'


def build_static(index_dates):
    """Patient-level fields; `index_dates` supplies the reference point for Age and duration."""
    demo = pd.read_csv(DATA_DIR / FILES['demographics'])
    demo['RACE'] = demo.apply(infer_race, axis=1)
    demo['BIRTHDT'] = pd.to_datetime(demo['BIRTHDT'], format='%m/%Y', errors='coerce')
    demo = demo.groupby('PATNO', as_index=False)[['BIRTHDT', 'SEX', 'RACE']].first()

    fam = pd.read_csv(DATA_DIR / FILES['family'])
    fam = fam.groupby('PATNO', as_index=False)['ANYFAMPD'].max()
    fam['FamilyHistory'] = fam['ANYFAMPD'].gt(0).astype(float).where(fam['ANYFAMPD'].notna())
    fam = fam[['PATNO', 'FamilyHistory']]

    dx = pd.read_csv(DATA_DIR / FILES['diagnosis'])
    dx = dx.groupby('PATNO', as_index=False)[
        ['PDDXDT', 'DXTREMOR', 'DXRIGID', 'DXBRADY', 'DXPOSINS', 'DOMSIDE']
    ].first()
    dx['PDDXDT'] = pd.to_datetime(dx['PDDXDT'], format='%m/%Y', errors='coerce')

    out = (
        index_dates[['PATNO', 'index_date']]
        .merge(demo, on='PATNO', how='left')
        .merge(fam, on='PATNO', how='left')
        .merge(dx, on='PATNO', how='left')
    )

    # Age and disease duration as of the index date, not as of today.
    out['Age'] = (
        out['index_date'].dt.year - out['BIRTHDT'].dt.year
        - (out['index_date'].dt.month < out['BIRTHDT'].dt.month).astype(int)
    )
    out['No_of_years'] = (out['index_date'] - out['PDDXDT']).dt.days / 365.25

    return out.drop(columns=['BIRTHDT', 'PDDXDT'])

## Time-varying predictors

All eleven longitudinal sources go through the same rule. Three need a per-visit value computed before the temporal filter can be applied:

- **GDS-15** — five items are reverse-scored, then summed per visit. Legacy took the *max* total across all visits; here it is the last total at or before the index date.
- **BMI** — computed per visit from height and weight. Values outside 10–60 kg/m² are treated as missing rather than repaired (a BMI of 300 is a data-entry error; the legacy overwrite with 25.06 invents a plausible-looking number). Adult height is then reconciled within each patient: a recorded height that is implausible (<140 or >210 cm) or >10 cm from that patient's consensus adult height is replaced with the consensus height before BMI is computed. Weights that deviate >50% from the patient's typical weight (median among visits with BMI 15–40) are voided (extra digit, lb-as-kg). Missing height is *not* filled — we only correct a height that was written down wrong. See `data_cleaning.md`.
- **Freezing of gait** — comes from the falls table itself, so the date filter is what stops the outcome visit from leaking in. `FRZGT12M` also has a 12-month recall window, and the index date keeps that window from overlapping the outcome's. Patients with no falls visit at or before the index date legitimately get `NaN`.

`MDS-UPDRS Part IV` was previously the max across all visits, which could be drawn from after the outcome. It is now the last value at or before the index date like everything else.

In [5]:
GDS_FLIP = ['GDSSATIS', 'GDSGSPIR', 'GDSHAPPY', 'GDSALIVE', 'GDSENRGY']
GDS_ITEMS = [
    'GDSSATIS', 'GDSDROPD', 'GDSEMPTY', 'GDSBORED', 'GDSGSPIR', 'GDSAFRAD',
    'GDSHAPPY', 'GDSHLPLS', 'GDSHOME', 'GDSMEMRY', 'GDSALIVE', 'GDSWRTLS',
    'GDSENRGY', 'GDSHOPLS', 'GDSBETER',
]

RENAME = {
    'DOPTHERST': 'Dopaminergic therapy started for participant',
    'MCATOT': 'MoCA Total Score',
    'FRZGT12M': 'Freezing of gait (peak severity)',
    'SCAU14': 'lightheaded after standing',
    'SCAU16': 'fainted',
    'NP1RTOT': 'MDS-UPDRS Part I Score',
    'NP1SLPD': 'Daytime_Sleepiness',
    'NP1URIN': 'Urinary_Problems',
    'NP3TOT': 'MDS-UPDRS Part III Score',
    'NP3GAIT': 'Gait',
    'NP3PSTBL': 'Postural_Stability',
    'NHY': 'Hoehn_And_Yahr_Stage',
    'NP4TOT': 'MDS-UPDRS PartIV score',
    'FEATPOSHYP': 'Postural_hypotension',
    'DXRIGID': 'Rigidity present at diagnosis?',
    'DXPOSINS': 'Postural instability present at dx?',
    'DXTREMOR': 'Resting Tremor present at diagnosis?',
    'DXBRADY': 'Bradykinesia present at diagnosis?',
    'DOMSIDE': 'Side predominantly affected at onset',
    'delta_MCATOT': 'Delta MoCA',
    'delta_NP1RTOT': 'Delta UPDRS I',
    'delta_NP3TOT': 'Delta UPDRS III',
    'delta_NP4TOT': 'Delta UPDRS IV',
    'delta_GDS_TOTAL': 'Delta Depression',
    'delta_NP3GAIT': 'Delta Gait',
    'delta_NP3PSTBL': 'Delta Postural Stability',
    'delta_NHY': 'Delta H&Y',
    'delta_BMI': 'Delta BMI',
    'delta_able_weighted_score': 'Delta Neuro-QoL',
}

def gds_visits():
    df = load_visits('gds')
    for c in GDS_FLIP:
        df[c] = df[c].where(df[c].isna(), 1 - df[c].astype(float))
    df['GDS_TOTAL'] = df[GDS_ITEMS].sum(axis=1, min_count=1)
    return df

_BMI_CACHE = None
_BMI_STATS = {}

def _consensus_height(heights, ht_lo=140, ht_hi=210):
    """Median of adult-plausible heights, or the majority 5 cm cluster if they disagree."""
    h = heights.dropna()
    h = h[(h >= ht_lo) & (h <= ht_hi)]
    if h.empty:
        return np.nan
    if h.max() - h.min() <= 10:
        return float(h.median())
    bins = (h / 5).round() * 5
    top = bins.value_counts()
    winners = top[top == top.max()].index
    if len(winners) == 1:
        return float(h[bins == winners[0]].median())
    return np.nan


def bmi_visits():
    """Per-visit BMI with visit-level typo cleanup (not a patient drop, not a mean fill)."""
    global _BMI_CACHE, _BMI_STATS
    if _BMI_CACHE is not None:
        return _BMI_CACHE.copy()

    df = load_visits('vitals')
    df['HTCM'] = pd.to_numeric(df['HTCM'], errors='coerce')
    df['WGTKG'] = pd.to_numeric(df['WGTKG'], errors='coerce')

    ht_lo, ht_hi = 140, 210
    n_wt_void = n_ht_fix = 0
    parts = []
    for _, g in df.groupby('PATNO'):
        g = g.copy()
        raw_bmi = g['WGTKG'] / (g['HTCM'] / 100.0) ** 2
        typical = g['HTCM'].between(ht_lo, ht_hi) & raw_bmi.between(15, 40)
        typ_w = g.loc[typical, 'WGTKG'].median()
        if pd.isna(typ_w):
            typ_w = g['WGTKG'].median()
        w_ok = g['WGTKG'].isna() | pd.isna(typ_w) | ((g['WGTKG'] - typ_w).abs() / typ_w <= 0.5)
        n_wt_void += int((~w_ok & g['WGTKG'].notna()).sum())

        ht_cons = _consensus_height(g['HTCM'], ht_lo, ht_hi)
        if pd.isna(ht_cons):
            h = g.loc[g['HTCM'].between(ht_lo, ht_hi), 'HTCM']
            if not h.empty:
                bins = (h / 5).round() * 5
                best, best_n = np.nan, -1
                for b in bins.unique():
                    hc = float(h[bins == b].median())
                    n = int((g['WGTKG'] / (hc / 100.0) ** 2).between(18, 35).sum())
                    if n > best_n:
                        best, best_n = hc, n
                ht_cons = best

        ht_use = g['HTCM'].copy()
        if pd.notna(ht_cons):
            # Correct a recorded-but-wrong height. Do not invent height on blank rows.
            bad_ht = g['HTCM'].notna() & (
                ~g['HTCM'].between(ht_lo, ht_hi) | ((g['HTCM'] - ht_cons).abs() > 10)
            )
            n_ht_fix += int((bad_ht & g['WGTKG'].notna()).sum())
            ht_use = ht_use.where(~bad_ht, ht_cons)

        bmi = g['WGTKG'] / (ht_use / 100.0) ** 2
        g['BMI'] = bmi.where(w_ok).where(bmi.between(10, 60))
        parts.append(g)

    out = pd.concat(parts, ignore_index=True)
    _BMI_STATS = {'weight_voids': n_wt_void, 'height_fixes': n_ht_fix}
    _BMI_CACHE = out
    return out.copy()

def source_tables():
    return [
        ('Dopaminergic therapy', load_visits('dopamine'),  ['DOPTHERST']),
        ('MoCA',                 load_visits('moca'),      ['MCATOT']),
        ('Freezing/falls',       falls,                    ['FRZGT12M']),
        ('SCOPA-AUT',            load_visits('scopa'),     ['SCAU14', 'SCAU16']),
        ('Neuro-QoL',            load_visits('neuroqol'),  ['NQMOB37', 'NQMOB30', 'NQMOB26',
                                                            'NQMOB32', 'NQMOB33', 'NQMOB31',
                                                            'NQMOB28', 'NQMOB25']),
        ('GDS-15',               gds_visits(),             ['GDS_TOTAL']),
        ('UPDRS Part I',         load_visits('updrs1'),    ['NP1RTOT']),
        ('UPDRS Part I quest.',  load_visits('updrs1q'),   ['NP1SLPD', 'NP1URIN']),
        ('UPDRS Part III',       load_visits('updrs3'),    ['NP3TOT', 'NP3GAIT',
                                                            'NP3PSTBL', 'NHY']),
        ('UPDRS Part IV',        load_visits('updrs4'),    ['NP4TOT']),
        ('Vital signs (BMI)',    bmi_visits(),             ['BMI']),
        ('Other clinical',       load_visits('otherclin'), ['FEATPOSHYP']),
    ]

def first_visit_dates():
    firsts = []
    for _, visits, cols in source_tables():
        m = visits[visits['DATE'].notna() & visits[cols].notna().any(axis=1)]
        firsts.append(m.groupby('PATNO')['DATE'].min())
    return (pd.concat(firsts).groupby(level=0).min()
            .rename('first_visit_date').reset_index())

def build_timevarying(index_dates, inclusive=True):
    out = index_dates[['PATNO']].copy()
    for _, visits, cols in source_tables():
        numeric = visits.copy()
        numeric[cols] = numeric[cols].apply(pd.to_numeric, errors='coerce')
        
        # Fix FEATPOSHYP (2 = Uncertain -> NaN)
        if 'FEATPOSHYP' in cols:
            numeric['FEATPOSHYP'] = numeric['FEATPOSHYP'].replace(2, np.nan)
        
        for c in list(cols):
            if c in ['NP3GAIT', 'NP3PSTBL', 'NHY']:
                sentinel_col = f'{c}_missing_due_to_101'
                numeric[sentinel_col] = np.where(numeric[c].isna(), np.nan, (numeric[c] == 101).astype(float))
                numeric[c] = numeric[c].replace(101, np.nan)
                if sentinel_col not in cols:
                    cols.append(sentinel_col)
                    
        # Calculate able_weighted_score BEFORE aggregation
        able_raw = ['NQMOB37', 'NQMOB30', 'NQMOB26', 'NQMOB32', 'NQMOB33', 'NQMOB31', 'NQMOB28', 'NQMOB25']
        if all(c in cols for c in able_raw):
            mu, sigma = 2.5, 1.2
            weighted = numeric[able_raw].copy()
            for c in able_raw:
                weighted[c] = np.exp(-((weighted[c] - mu)**2) / (2 * sigma**2))
            
            # Use mean * 8 to normalize partial responses onto the full scale
            numeric['able_weighted_score'] = weighted.mean(axis=1) * 8
            cols.append('able_weighted_score')
            for c in able_raw:
                cols.remove(c)
                
        out = out.merge(
            latest_at_index(numeric, cols, index_dates, inclusive),
            on='PATNO', how='left',
        )

    return out.rename(columns={**RENAME, 'GDS_TOTAL': 'Total Depression Score'})


## Assemble a dataset variant

`build_dataset` glues the pieces together for one index-date offset and applies the one exclusion the professor asked for: a patient with **no predictor data at all at or before the index date** cannot be predicted prospectively and is dropped. Patients missing only *some* predictors are kept and handled by imputation later.

The 27 modelling features keep their legacy names so the explanation layer's `feature_map.csv` still lines up — except `able_weighted_score`, which is gone, replaced by the seven raw "Able to" items it was built from.

In [6]:
FEATURES = [
    'No_of_years', 'Age',
    'Postural instability present at dx?', 'Rigidity present at diagnosis?',
    'Dopaminergic therapy started for participant',
    'MoCA Total Score',
    'Freezing of gait (peak severity)',
    'lightheaded after standing', 'fainted',
    'Total Depression Score',
    'able_weighted_score',
    'MDS-UPDRS Part I Score',
    'BMI',
    'Daytime_Sleepiness', 'Urinary_Problems',
    'Gait', 'Postural_Stability', 'Hoehn_And_Yahr_Stage',
    'Postural_hypotension',
    'MDS-UPDRS Part III Score',
    'MDS-UPDRS PartIV score',
    'NP3GAIT_missing_due_to_101', 'NP3PSTBL_missing_due_to_101', 'NHY_missing_due_to_101',
    'Delta MoCA', 'Delta UPDRS I', 'Delta UPDRS III', 'Delta UPDRS IV',
    'Delta Depression', 'Delta Gait', 'Delta Postural Stability', 'Delta H&Y',
    'Delta BMI', 'Delta Neuro-QoL'
]

META = ['PATNO', 'index_date', 'outcome_date', 'first_visit_date', 'history_months',
        'n_falls_visits', 'SEX', 'RACE', 'falls_raw', 'falls_class']

FIRST_VISITS = first_visit_dates()

STATIC_FEATURES = ['Age', 'No_of_years',
                   'Postural instability present at dx?', 'Rigidity present at diagnosis?']
VISIT_FEATURES = [c for c in FEATURES if c not in STATIC_FEATURES and not c.startswith('Delta ')]

def build_dataset(offset_months=OUTCOME_WINDOW_MONTHS):
    index_dates = index_dates_for(outcome_visits, offset_months)
    static = build_static(index_dates)
    timevarying = build_timevarying(index_dates, inclusive=offset_months > 0)

    df = (
        outcome_visits[['PATNO', 'outcome_date', 'n_falls_visits', 'falls_raw', 'falls_class']]
        .merge(index_dates[['PATNO', 'index_date']], on='PATNO', how='left')
        .merge(static.drop(columns=['index_date']), on='PATNO', how='left')
        .merge(timevarying, on='PATNO', how='left')
        .merge(FIRST_VISITS, on='PATNO', how='left')
        .rename(columns=RENAME)
    )
    df['history_months'] = ((df['index_date'] - df['first_visit_date']).dt.days / 30.44).round(1)

    no_data = df[VISIT_FEATURES].isna().all(axis=1)
    excluded = df[no_data]
    df = df[~no_data].reset_index(drop=True)

    return df, excluded


## Build both variants

In [7]:
VARIANTS = {'landmark': OUTCOME_WINDOW_MONTHS, 'no_landmark': 0}
PRIMARY = 'landmark'

datasets, dropped = {}, {}
for name, offset in VARIANTS.items():
    datasets[name], dropped[name] = build_dataset(offset)
    df = datasets[name]
    print(f'{name:<12} {len(df):>5} patients  |  {len(dropped[name])} excluded (no data at or before index)'
          f'  |  mean missing per patient: {df[FEATURES].isna().sum(axis=1).mean():.1f}/{len(FEATURES)}')

datasets[PRIMARY].head()

landmark      1040 patients  |  240 excluded (no data at or before index)  |  mean missing per patient: 1.7/34


no_landmark   1051 patients  |  229 excluded (no data at or before index)  |  mean missing per patient: 1.1/34


,PATNO,outcome_date,n_falls_visits,falls_raw,falls_class,index_date,SEX,RACE,FamilyHistory,Resting Tremor present at diagnosis?,Rigidity present at diagnosis?,Bradykinesia present at diagnosis?,Postural instability present at dx?,Side predominantly affected at onset,Age,No_of_years,Dopaminergic therapy started for participant,MoCA Total Score,Delta MoCA,Freezing of gait (peak severity),lightheaded after standing,fainted,able_weighted_score,Delta Neuro-QoL,Total Depression Score,Delta Depression,MDS-UPDRS Part I Score,Delta UPDRS I,Daytime_Sleepiness,Urinary_Problems,MDS-UPDRS Part III Score,Gait,Postural_Stability,Hoehn_And_Yahr_Stage,NP3GAIT_missing_due_to_101,NP3PSTBL_missing_due_to_101,NHY_missing_due_to_101,Delta UPDRS III,Delta Gait,Delta Postural Stability,Delta H&Y,MDS-UPDRS PartIV score,Delta UPDRS IV,BMI,Delta BMI,Postural_hypotension,first_visit_date,history_months
0,3003,2026-05-01,6,0.0,0,2025-05-01,0.0,White,0.0,0.0,1.0,1.0,1.0,2.0,70,16.167009,1.0,28.0,1.0,1.0,2.0,0.0,2.403331,0.459022,1.0,0.0,5.0,-2.0,2.0,3.0,59.0,2.0,3.0,3.0,0.0,0.0,0.0,-19.0,-1.0,-1.0,0.0,8.0,-1.0,23.999459,1.428412,1.0,2011-03-01,170.0
1,3010,2026-06-01,7,2.0,2,2025-06-01,1.0,White,1.0,0.0,1.0,1.0,0.0,1.0,61,14.329911,0.0,29.0,4.0,4.0,1.0,0.0,3.893368,0.802694,11.0,1.0,13.0,5.0,3.0,3.0,58.0,2.0,1.0,3.0,0.0,0.0,0.0,15.0,0.0,1.0,1.0,10.0,2.0,29.168692,-0.603536,1.0,2011-05-01,169.0
2,3018,2026-04-01,7,2.0,2,2025-04-01,0.0,White,1.0,1.0,1.0,2.0,0.0,2.0,73,13.248460,NaN,28.0,3.0,3.0,1.0,0.0,3.778017,0.802694,6.0,1.0,6.0,0.0,3.0,4.0,47.0,3.0,2.0,4.0,0.0,1.0,0.0,-2.0,0.0,0.0,0.0,5.0,4.0,26.838648,0.000000,1.0,2012-02-01,158.0
3,3020,2019-04-01,1,1.0,1,2018-04-01,0.0,White,NaN,1.0,2.0,1.0,0.0,2.0,79,6.247775,NaN,23.0,1.0,NaN,1.0,0.0,NaN,NaN,7.0,3.0,6.0,-1.0,2.0,3.0,45.0,1.0,3.0,3.0,0.0,0.0,0.0,-4.0,-1.0,1.0,0.0,0.0,NaN,31.364531,-0.694418,0.0,2012-03-01,73.0
4,3021,2026-04-01,7,2.0,2,2025-04-01,0.0,White,0.0,1.0,1.0,1.0,0.0,2.0,77,13.163587,1.0,29.0,4.0,3.0,1.0,0.0,3.090674,-1.949059,1.0,-1.0,6.0,-2.0,3.0,4.0,40.0,3.0,3.0,4.0,0.0,1.0,0.0,-18.0,0.0,0.0,0.0,2.0,2.0,32.894737,2.113829,1.0,2012-03-01,157.0


## Diagnostics

Three things the paper needs: how many falls visits each patient contributes, how stale the predictors are relative to the outcome visit, and how much is missing per feature. The last table is the direct input to the Phase 3 imputation decisions.

In [8]:
main = datasets[PRIMARY]

visit_dist = (
    main['n_falls_visits'].value_counts().sort_index()
    .rename_axis('falls visits per patient').to_frame('patients')
)
visit_dist['%'] = (100 * visit_dist['patients'] / len(main)).round(1)
print('Patient visit distribution\n')
print(visit_dist)
print(f"\nIndex dates span    {main['index_date'].min():%Y-%m} to {main['index_date'].max():%Y-%m}")
print(f"Outcome visits span {main['outcome_date'].min():%Y-%m} to {main['outcome_date'].max():%Y-%m}")

HISTORY_BINS = [-0.01, 6, 12, 24, np.inf]
HISTORY_LABELS = ['< 6 mo', '6–12 mo', '1–2 y', '> 2 y']
history_bucket = pd.cut(main['history_months'], HISTORY_BINS, labels=HISTORY_LABELS)
print('\nHistory available at the index date (first recorded visit → index date)\n')
print(pd.crosstab(history_bucket, main['falls_class'].map(
    {0: 'No fall', 1: 'Rare fall', 2: 'Recurrent fall'}), margins=True, margins_name='All'))

n_missing = main[VISIT_FEATURES].isna().sum(axis=1)
print(f'\nMissing predictors per retained patient (of {len(VISIT_FEATURES)} visit-sourced features):')
print(n_missing.describe()[['mean', '50%', 'max']].round(1).to_string())
for threshold in (5, 10, 15):
    print(f'  patients missing more than {threshold}: {(n_missing > threshold).sum()}')

Patient visit distribution

                          patients     %
falls visits per patient                
1                               58   5.6
2                               87   8.4
3                              244  23.5
4                              237  22.8
5                              184  17.7
6                              162  15.6
7                               49   4.7
8                               19   1.8

Index dates span    2018-02 to 2025-08
Outcome visits span 2019-02 to 2026-08

History available at the index date (first recorded visit → index date)

falls_class     No fall  Rare fall  Recurrent fall   All
history_months                                          
< 6 mo               44         13               0    57
6–12 mo              57         10               0    67
1–2 y               131         23               6   160
> 2 y               480        181              95   756
All                 712        227             101  1040

Missing p

In [9]:
idx = index_dates_for(outcome_visits)

rows = []
for label, visits, cols in source_tables():
    m = visits.merge(idx[['PATNO', 'outcome_date', 'index_date']], on='PATNO', how='inner')
    m = m[m['DATE'].notna() & (m['DATE'] < m['outcome_date']) & m[cols].notna().any(axis=1)]
    last = m.sort_values(['PATNO', 'DATE']).groupby('PATNO').tail(1)
    gap = (last['outcome_date'] - last['DATE']).dt.days / 30.44
    rows.append({
        'source': label,
        'patients with any prior data': len(last),
        'patients with data at/before index date': m[m['DATE'] <= m['index_date']]['PATNO'].nunique(),
        'last visit sits inside follow-up window': int((last['DATE'] > last['index_date']).sum()),
        'gap median (mo)': round(gap.median(), 1),
        'gap p90 (mo)': round(gap.quantile(0.90), 1),
    })

gap_table = pd.DataFrame(rows).sort_values('patients with any prior data', ascending=False)
print(f'Predictor recency relative to the outcome visit (of {len(outcome_visits)} patients)\n')
gap_table

Predictor recency relative to the outcome visit (of 1280 patients)



,source,patients with any prior data,patients with data at/before index date,last visit sits inside follow-up window,gap median (mo),gap p90 (mo)
1,MoCA,1047,1040,319,12.0,18.0
11,Other clinical,1047,1040,638,11.0,14.0
3,SCOPA-AUT,1046,1040,334,12.0,15.0
5,GDS-15,1046,1040,331,12.0,15.5
6,UPDRS Part I,1046,1040,653,10.1,13.9
7,UPDRS Part I quest.,1046,1040,944,6.0,11.0
8,UPDRS Part III,1046,1040,576,11.0,14.0
10,Vital signs (BMI),1044,1040,304,12.0,23.0
2,Freezing/falls,987,977,321,12.0,15.0
4,Neuro-QoL,987,976,323,12.0,15.0


In [10]:
missing = pd.DataFrame({
    'feature': FEATURES,
    'missing % (landmark)': [
        round(100 * datasets['landmark'][f].isna().mean(), 1) for f in FEATURES],
    'missing % (no landmark)': [
        round(100 * datasets['no_landmark'][f].isna().mean(), 1) for f in FEATURES],
}).sort_values('missing % (landmark)', ascending=False).reset_index(drop=True)

print('Missing values per feature — input to the Phase 3 imputation strategy\n')
missing

Missing values per feature — input to the Phase 3 imputation strategy



,feature,missing % (landmark),missing % (no landmark)
0,Delta UPDRS IV,32.8,23.7
1,MDS-UPDRS PartIV score,21.7,14.9
2,Delta Neuro-QoL,20.3,14.6
3,Dopaminergic therapy started for participant,19.4,11.1
4,Delta BMI,13.8,9.0
5,Delta Depression,13.0,7.9
6,Delta MoCA,11.9,7.5
7,able_weighted_score,6.2,6.1
8,Freezing of gait (peak severity),6.1,6.1
9,Delta Postural Stability,5.5,1.2


## Sanity checks

The first two are the claims the reviewer response rests on, so they are asserted rather than eyeballed: no predictor value may come from inside the follow-up window, and the outcome must never be imputed.

In [11]:
for name, df in datasets.items():
    assert df['PATNO'].is_unique, f'{name}: duplicate patients'
    assert df['falls_class'].notna().all(), f'{name}: outcome must never be missing'
    assert set(df['falls_class']) <= {0, 1, 2}, f'{name}: unexpected outcome class'
    assert 'able_weighted_score' in df.columns

# No value may be drawn from inside the follow-up window. Sitting on the index date is allowed;
# anything later means falls had already begun accruing before the measurement.
for label, visits, cols in source_tables():
    m = visits.merge(idx[['PATNO', 'outcome_date', 'index_date']], on='PATNO', how='inner')
    used = m[m['DATE'].notna() & (m['DATE'] <= m['index_date'])]
    months_before_outcome = (used['outcome_date'] - used['DATE']).dt.days / 30.44
    assert (months_before_outcome >= OUTCOME_WINDOW_MONTHS - 0.5).all(), \
        f'{label}: value drawn from inside the follow-up window'

# Patients with a single falls visit have no prior freezing measurement by construction.
single = main[main['n_falls_visits'] == 1]
assert single['Freezing of gait (peak severity)'].isna().all()

print('BMI visit cleanup:', _BMI_STATS)
pi = main['Postural instability present at dx?']
print('PI at dx (should be DXPOSINS, Yes ~90 not ~187):')
print(pi.value_counts(dropna=False).to_string())
assert pi.eq(1).sum() < 150, 'PI-at-dx Yes count still looks like DXOTHSX'

print('\\nSpot-check BMI for previously flagged PATNOs:')
for pid in [157041, 201630, 219153, 137482, 167222, 143835]:
    row = main.loc[main['PATNO'] == pid, ['BMI', 'Delta BMI']]
    if row.empty:
        print(f'  {pid}: not in landmark')
    else:
        print(f'  {pid}: BMI={row["BMI"].iloc[0]:.2f}  ΔBMI={row["Delta BMI"].iloc[0]}')



BMI visit cleanup: {'weight_voids': 35, 'height_fixes': 128}
PI at dx (should be DXPOSINS, Yes ~90 not ~187):
Postural instability present at dx?
0.0    887
1.0     92
2.0     60
NaN      1
\nSpot-check BMI for previously flagged PATNOs:
  157041: BMI=23.77  ΔBMI=nan
  201630: BMI=21.54  ΔBMI=nan
  219153: BMI=33.02  ΔBMI=3.086419753086421
  137482: BMI=25.15  ΔBMI=0.04328254847645496
  167222: BMI=32.52  ΔBMI=1.761772522320765
  143835: BMI=22.15  ΔBMI=-1.3489435266148924


## Save

Datasets go to `revision_work/data/`. They still contain `NaN` — that is deliberate, and the next notebook is where imputation happens.

In [12]:
main = datasets['landmark']
ref = datasets['no_landmark']

delta_cols = [c for c in FEATURES if c.startswith('Delta ')]
base_features = [c for c in FEATURES if not c.startswith('Delta ')]

path = OUT_DIR / 'modelling_landmark_with_deltas.csv'
main[META + FEATURES].to_csv(path, index=False)
print(f'{path.name:<38} {main.shape[0]} x {len(META+FEATURES)}')

path = OUT_DIR / 'modelling_landmark.csv'
main[META + base_features].to_csv(path, index=False)
print(f'{path.name:<38} {main.shape[0]} x {len(META+base_features)}')

path = OUT_DIR / 'modelling_no_landmark.csv'
ref[META + base_features].to_csv(path, index=False)
print(f'{path.name:<38} {ref.shape[0]} x {len(META+base_features)}')

missing.to_csv(OUT_DIR / 'missingness_by_feature.csv', index=False)
gap_table.to_csv(OUT_DIR / 'predictor_recency.csv', index=False)
visit_dist.to_csv(OUT_DIR / 'visit_distribution.csv')
print('\nDiagnostic tables written to', OUT_DIR)


modelling_landmark_with_deltas.csv     1040 x 44
modelling_landmark.csv                 1040 x 34
modelling_no_landmark.csv              1051 x 34

Diagnostic tables written to /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/PD Prediction Old/Mobility_Decline_Risk_Prediction_For_PD_Patients-explanation_llm_v2/revision_work/data
